Copyright (c) Microsoft Corporation. All rights reserved.

Licensed under the MIT License.

![Impressions](https://PixelServer20190423114238.azurewebsites.net/api/impressions/MachineLearningNotebooks/how-to-use-azureml/work-with-data/datadrift-tutorial/datadrift-quickdemo.png)

# Analyze data drift in Azure Machine Learning datasets 

In this tutorial, you will setup a data drift monitor on a weather dataset to:

&#x2611; Analyze historical data for drift

&#x2611; Setup a monitor to recieve email alerts if data drift is detected going forward

If your workspace is Enterprise SKU, view and exlpore the results in the Azure Machine Learning studio. The video below shows the results from this tutorial (spoiler alert). 

## Prerequisites
If you are using an Azure Machine Learning Compute instance, you are all set. Otherwise, go through the [configuration notebook](../../../configuration.ipynb) first if you haven't already established your connection to the AzureML Workspace.

In [1]:
# Check core SDK version number
import azureml.core

print('SDK version:', azureml.core.VERSION)

Failure while loading azureml_run_type_providers. Failed to load entrypoint hyperdrive = azureml.train.hyperdrive:HyperDriveRun._from_run_dto with exception (azureml-core 1.0.65 (/home/cody/miniconda3/envs/aml/lib/python3.7/site-packages), Requirement.parse('azureml-core==0.1.0.5621271.*')).
Failure while loading azureml_run_type_providers. Failed to load entrypoint azureml.PipelineRun = azureml.pipeline.core.run:PipelineRun._from_dto with exception (azureml-core 1.0.65 (/home/cody/miniconda3/envs/aml/lib/python3.7/site-packages), Requirement.parse('azureml-core==0.1.0.5621271.*')).
Failure while loading azureml_run_type_providers. Failed to load entrypoint azureml.ReusedStepRun = azureml.pipeline.core.run:StepRun._from_reused_dto with exception (azureml-core 1.0.65 (/home/cody/miniconda3/envs/aml/lib/python3.7/site-packages), Requirement.parse('azureml-core==0.1.0.5621271.*')).
Failure while loading azureml_run_type_providers. Failed to load entrypoint azureml.StepRun = azureml.pipeli

SDK version: 1.0.65


## Initialize Workspace

Initialize a workspace object from persisted configuration.

In [2]:
from azureml.core import Workspace

ws = Workspace.from_config()

## Setup target and baseline datasets

Setup the baseline and target datasets. The baseline will be used to compare each time slice of the target dataset, which is sampled by a given frequency. For further details, see [our documentation](http://aka.ms/datadrift). 

The next few cells will:
  * get the default datastore
  * upload the `weather-data` to the datastore
  * create the Tabular dataset from the data
  * add the timeseries trait by specifying the timestamp column `datetime`
  * register the dataset
  * create the baseline as a time slice of the target dataset
  * optionally, register the baseline dataset
  
The folder `weather-data` contains weather data from the [NOAA Integrated Surface Data](https://azure.microsoft.com/services/open-datasets/catalog/noaa-integrated-surface-data/) filtered down to to station names containing the string 'FLORIDA' to reduce the size of data. See `get_data.py` to see how this data is curated and modify as desired. This script may take a long time to run, hence the data is provided in the `weather-data` folder for this demo.

In [3]:
# use default datastore
dstore = ws.get_default_datastore()

In [4]:
# upload weather data
dstore.upload('weather-data', 'datadrift-data', overwrite=True, show_progress=False);

UserErrorException: UserErrorException:
	Message: src_path must be a directory.
	InnerException None
	ErrorResponse 
{
    "error": {
        "code": "UserError",
        "message": "src_path must be a directory."
    }
}

In [5]:
# import Dataset class
from azureml.core import Dataset

# create dataset 
dset = Dataset.Tabular.from_parquet_files(dstore.path('datadrift-data/**/data.parquet'))
dset = dset.with_timestamp_columns('datetime')

In [6]:
# import datetime 
from datetime import datetime

# naively train a model on first 3 months of weather data
training = dset.time_between(datetime(2019, 1, 1), datetime(2019, 2, 1))
training = training.register(ws, 'noaa-training20')

2019-10-15 05:40:12.045922 | ActivityCompleted: Activity=register, HowEnded=Failure, Duration=6366.61 [ms], Info = {'activity_id': '22ce599b-6e3d-4e6e-89d4-52a5d2aa2294', 'activity_name': 'register', 'activity_type': 'PublicApi', 'app_name': 'dataset', 'source': 'azureml.dataset', 'version': '1.0.65', 'completionStatus': 'Success', 'durationMs': 3120.53}, Exception=HttpOperationError; Operation returned an invalid status code "Definition '1' has been used in version '1' of an existing registered dataset named 'noaa-training11"


HttpOperationError: Operation returned an invalid status code "Definition '1' has been used in version '1' of an existing registered dataset named 'noaa-training11"

In [7]:
training = Dataset.get_by_name(ws, 'noaa-training11')

In [8]:
from sklearn.linear_model import LinearRegression

In [9]:
df = training.to_pandas_dataframe()

/home/cody/miniconda3/envs/aml/lib/python3.7/site-packages/azureml/dataprep/api/dataflow.py:681: UserWarning: Inconsistent or mixed schemas detected across partitions. Using alternate reader.
  warnings.warn('Inconsistent or mixed schemas detected across partitions. Using alternate reader.')


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4041 entries, 0 to 4040
Data columns (total 23 columns):
usaf                       4041 non-null object
wban                       4041 non-null object
datetime                   4041 non-null datetime64[ns]
latitude                   4041 non-null float64
longitude                  4041 non-null float64
elevation                  4041 non-null float64
windAngle                  3897 non-null float64
windSpeed                  3948 non-null float64
temperature                3975 non-null float64
seaLvlPressure             2751 non-null float64
cloudCoverage              1705 non-null object
presentWeatherIndicator    875 non-null float64
pastWeatherIndicator       469 non-null float64
precipTime                 2595 non-null float64
precipDepth                2595 non-null float64
snowDepth                  75 non-null float64
stationName                4041 non-null object
countryOrRegion            4041 non-null object
p_k          

In [11]:
df = df.fillna(0)

In [12]:
X_features = ['latitude', 'longitude', 'temperature', 'windAngle', 'windSpeed']
y_features = ['elevation']

In [13]:
X = df[X_features]
y = df[y_features]

In [14]:
model = LinearRegression().fit(X, y)

In [15]:
model.predict([[30, -85, 21, 150, 6]])

array([[3.14320305]])

In [16]:
import pickle

with open('model.pkl', 'wb') as f:
    pickle.dump(model, f)

In [17]:
from azureml.core.model import Model
#from azureml.core.resource_configuration import ResourceConfiguration

model = Model.register(model_path="model.pkl",
                       model_name="model.pkl",
                       tags={'area': "weather", 'type': "linear regression"},
                       description="Linear regression model to predict elevation based on the weather",
                       workspace=ws)

Registering model model.pkl


In [18]:
from azureml.core import Environment

env = Environment.from_conda_specification(name='deploytocloudenv', file_path='myenv.yml')

In [19]:
from azureml.core.model import InferenceConfig

inference_config = InferenceConfig(entry_script="score.py", environment=env)

In [20]:
from azureml.core.compute import AksCompute, ComputeTarget

# Use the default configuration (you can also provide parameters to customize this).
# For example, to create a dev/test cluster, use:
# prov_config = AksCompute.provisioning_configuration(cluster_purpose = AksCompute.ClusterPurpose.DEV_TEST)
prov_config = AksCompute.provisioning_configuration()

aks_name = 'myaks11'
# Create the cluster
try:
    aks_target = ws.compute_targets[aks_name]
except:
    aks_target = ComputeTarget.create(workspace = ws,
                                      name = aks_name,
                                      provisioning_configuration = prov_config)

    # Wait for the create process to complete
    aks_target.wait_for_completion(show_output = True)

In [27]:
from azureml.core.webservice import AksWebservice, Webservice
from azureml.exceptions import WebserviceException

deployment_config = AksWebservice.deploy_configuration(cpu_cores=1, memory_gb=1)
service_name = 'service21'

service = Model.deploy(ws, service_name, [model], inference_config, deployment_config, aks_target)

service.wait_for_deployment(True)
print(service.state)

Running...........
SucceededAKS service creation operation finished, operation "Succeeded"
Healthy


In [ ]:
print(service.get_logs())

In [23]:
import json
test_sample = json.dumps({'data': [
    [1,2,3,4,5], 
    [5,4,3,2,1]
]})

test_sample_encoded = bytes(test_sample, encoding='utf8')
prediction = service.run(input_data=test_sample_encoded)
print(prediction)

[[128.18424974977881], [120.60601173866114]]


In [24]:
ws.webservices['service20'].delete()

2019-10-15 05:58:36.801751 | ActivityCompleted: Activity=get_by_id, HowEnded=Failure, Duration=5396.69 [ms], Info = {'activity_id': 'bf268757-e08e-45eb-9c45-21186aab1200', 'activity_name': 'get_by_id', 'activity_type': 'PublicApi', 'app_name': 'dataset', 'source': 'azureml.dataset', 'version': '1.0.65', 'completionStatus': 'Success', 'durationMs': 5983.31}, Exception=HttpOperationError; Operation returned an invalid status code "Exception of type 'Common.WebApi.Exceptions.ResourceNotFoundException' was thrown."
WARNING - Cannot find Saved Dataset with ID 16338600-b419-4b03-8354-71091786194b due to the following exception.
Operation returned an invalid status code "Exception of type 'Common.WebApi.Exceptions.ResourceNotFoundException' was thrown."
2019-10-15 05:58:48.092141 | ActivityCompleted: Activity=get_by_id, HowEnded=Failure, Duration=5540.87 [ms], Info = {'activity_id': 'd3a2a6ed-57a2-4dc2-8696-445ff8aae6a4', 'activity_name': 'get_by_id', 'activity_type': 'PublicApi', 'app_name':

In [25]:
ws.webservices['service20']

2019-10-15 15:02:59.926333 | ActivityCompleted: Activity=get_by_id, HowEnded=Failure, Duration=447.75 [ms], Info = {'activity_id': '52442162-bb75-4315-9a4e-5695563248fa', 'activity_name': 'get_by_id', 'activity_type': 'PublicApi', 'app_name': 'dataset', 'source': 'azureml.dataset', 'version': '1.0.65', 'completionStatus': 'Success', 'durationMs': 667.4}, Exception=HttpOperationError; Operation returned an invalid status code "Exception of type 'Common.WebApi.Exceptions.ResourceNotFoundException' was thrown."
WARNING - Cannot find Saved Dataset with ID 16338600-b419-4b03-8354-71091786194b due to the following exception.
Operation returned an invalid status code "Exception of type 'Common.WebApi.Exceptions.ResourceNotFoundException' was thrown."
2019-10-15 15:03:01.086304 | ActivityCompleted: Activity=get_by_id, HowEnded=Failure, Duration=440.98 [ms], Info = {'activity_id': '00f59d72-1a18-48a5-883c-c2d0bf4c60b3', 'activity_name': 'get_by_id', 'activity_type': 'PublicApi', 'app_name': 'da

KeyError: 'service20'

In [26]:
ws.webservices

2019-10-15 15:03:15.069551 | ActivityCompleted: Activity=get_by_id, HowEnded=Failure, Duration=372.93 [ms], Info = {'activity_id': '8bb22f23-f77f-4311-b913-8e96c2fb413f', 'activity_name': 'get_by_id', 'activity_type': 'PublicApi', 'app_name': 'dataset', 'source': 'azureml.dataset', 'version': '1.0.65', 'completionStatus': 'Success', 'durationMs': 499.67}, Exception=HttpOperationError; Operation returned an invalid status code "Exception of type 'Common.WebApi.Exceptions.ResourceNotFoundException' was thrown."
WARNING - Cannot find Saved Dataset with ID 16338600-b419-4b03-8354-71091786194b due to the following exception.
Operation returned an invalid status code "Exception of type 'Common.WebApi.Exceptions.ResourceNotFoundException' was thrown."
2019-10-15 15:03:16.244590 | ActivityCompleted: Activity=get_by_id, HowEnded=Failure, Duration=435.36 [ms], Info = {'activity_id': '91995f02-5dd0-4434-9b97-352b2b698ca3', 'activity_name': 'get_by_id', 'activity_type': 'PublicApi', 'app_name': 'd

{'service13': AksWebservice(workspace=Workspace.create(name='Azure-ML', subscription_id='60582a10-b9fd-49f1-a546-c4194134bba8', resource_group='copetersRG'), name=service13, image_id=None, compute_type=AKS, state=None, scoring_uri=http://13.66.39.137:80/api/v1/service/service13/score, tags={}, properties={'azureml.git.repository_uri': 'https://github.com/lostmygithubaccount/dkdc.git', 'mlflow.source.git.repoURL': 'https://github.com/lostmygithubaccount/dkdc.git', 'azureml.git.branch': 'master', 'mlflow.source.git.branch': 'master', 'azureml.git.commit': 'c6b43516c75f48fbccf10d17d3fb744a57dcfb68', 'mlflow.source.git.commit': 'c6b43516c75f48fbccf10d17d3fb744a57dcfb68', 'azureml.git.dirty': 'True'}),
 'service12': AksWebservice(workspace=Workspace.create(name='Azure-ML', subscription_id='60582a10-b9fd-49f1-a546-c4194134bba8', resource_group='copetersRG'), name=service12, image_id=None, compute_type=AKS, state=None, scoring_uri=http://13.66.39.137:80/api/v1/service/service12/score, tags={}

In [ ]:
from azureml.core.model import Model

model = Model.register(ws, model_path='model.pkl', model_name='drift-model-temp', tags={'data drift': 'True'}, description='dummy model', datasets=[('training', training)])

In [ ]:
from azureml.core.compute import AksCompute, ComputeTarget

# Use the default configuration (you can also provide parameters to customize this).
# For example, to create a dev/test cluster, use:
# prov_config = AksCompute.provisioning_configuration(cluster_purpose = AksCompute.ClusterPurpose.DEV_TEST)
prov_config = AksCompute.provisioning_configuration()

aks_name = 'myaks'
# Create the cluster
aks_target = ComputeTarget.create(workspace = ws,
                                    name = aks_name,
                                    provisioning_configuration = prov_config)

# Wait for the create process to complete
aks_target.wait_for_completion(show_output = True)

In [ ]:
X.iloc[1]

In [ ]:
from azureml.core import Environment
from azureml.core.model import InferenceConfig

env = Environment.from_conda_specification('default', '../aml.yml')

inference_config = InferenceConfig(entry_script='score.py', environment=env)

In [ ]:
from azureml.core.webservice import AksWebservice, Webservice
from azureml.core.model import Model

aks_target = AksCompute(ws,"myaks")
# If deploying to a cluster configured for dev/test, ensure that it was created with enough
# cores and memory to handle this deployment configuration. Note that memory is also used by
# things such as dependencies and AML components.
deployment_config = AksWebservice.deploy_configuration(cpu_cores = 1, memory_gb = 1)
service = Model.deploy(ws, "myservice2", [model], inference_config, deployment_config, aks_target)
service.wait_for_deployment(show_output = True)
print(service.state)
print(service.get_logs())

In [ ]:
service.get_logs()

In [ ]:
help(service)

## Create data drift monitor

See [our documentation](http://aka.ms/datadrift) for a complete description for all of the parameters. 

In [ ]:
from azureml.datadrift import DataDriftDetector, AlertConfiguration

alert_config = AlertConfiguration(['user@contoso.com']) # replace with your email to recieve alerts from the scheduled pipeline after enabling

monitor = DataDriftDetector.create_from_datasets(ws, 'weather-monitor', baseline, target, 
                                                      compute_target_name='cpu-cluster',    # compute target for scheduled pipeline and backfills 
                                                      frequency='Week',                     # how often to analyze target data
                                                      feature_list=None,                    # list of features to detect drift on
                                                      drift_threshold=None,                 # threshold from 0 to 1 for email alerting
                                                      latency=0,                            # SLA in hours for target data to arrive in the dataset
                                                      alert_config=alert_config)            # email addresses to send alert

## Update data drift monitor

Many settings of the data drift monitor can be updated after creation. In this demo, we will update the `drift_threshold` and `feature_list`. See [our documentation](http://aka.ms/datadrift) for details on which settings can be changed.

In [ ]:
from azureml.datadrift import DataDriftDetector

# get monitor by name
monitor = DataDriftDetector.get_by_name(ws, 'weather-monitor')

In [ ]:
# create feature list - need to exclude columns that naturally drift or increment over time, such as year, day, index
columns  = list(baseline.take(1).to_pandas_dataframe())
exclude  = ['year', 'day', 'version', '__index_level_0__']
features = [col for col in columns if col not in exclude]

# update the feature list
monitor  = monitor.update(feature_list=features)

## Analyze historical data and backfill

You can use the `backfill` method to:
  * analyze historical data
  * backfill metrics after updating the settings (mainly the feature list)
  * backfill metrics for failed runs
  
The below cells will run two backfills that will produce data drift results for 2019 weather data, with January used as the baseline in the monitor. The output can be seen from the `show` method after the runs have completed, or viewed from the Azure Machine Learning studio for Enterprise workspaces.

![Drift results](media/drift-results.png)

>**Tip!** When starting with the data drift capability, start by backfilling on a small section of data to get initial results. Update the feature list as needed by removing columns that are causing drift, but can be ignored, and backfill this section of data until satisfied with the results. Then, backfill on a larger slice of data and/or set the alert configuration, threshold, and enable the schedule to recieve alerts to drift on your dataset. All of this can be done through the UI (Enterprise) or Python SDK.

Although it depends on many factors, the below backfills should typically take less than 20 minutes to run. Results will show as soon as they become available, not when the backfill is completed, so you may begin to see some metrics in a few minutes.

In [ ]:
# backfill for 1st half of the year
backfill1 = monitor.backfill(datetime(2019, 1, 1), datetime(2019, 7, 1))
backfill1

In [ ]:
# backfill for 2nd half of the year 
backfill2 = monitor.backfill(datetime(2019, 7, 1), datetime(2019, 10, 1))
backfill2

## Enable the monitor's pipeline schedule

Turn on a scheduled pipeline which will anlayze the target dataset for drift every `frequency`. Use the latency parameter to adjust the start time of the pipeline. For instance, if it takes 24 hours for my data processing pipelines for data to arrive in the target dataset, set latency to 24. 

In [ ]:
# enable the pipeline schedule and recieve email alerts
monitor = monitor.enable_schedule()

In [ ]:
# disable the pipeline schedule 
monitor = monitor.disable_schedule()

## Query metrics and show results in Python

The below cell will plot some key data drift metrics, and can be used to query the results. Run `help(monitor.get_output)` for specifics on the object returned.

In [ ]:
# get results from Python SDK (wait for backfills or monitor runs to finish)
#results, metrics = monitor.get_output()
#monitor.show()

## See results in Azure Machine Learning studio (Enterprise only)

The below cell will print a link to the monitor in the Azure Machine Learning studio, where the results can be viewed. Alertnatively, use the `show` or `get_results` to get and plot data drift results in Python.

In [ ]:
link = 'https://ml.azure.com/data/monitor/{}?wsid=/subscriptions/{}/resourcegroups/{}/workspaces/{}'.format(monitor.name, ws.subscription_id, ws.resource_group, ws.name)
print(link)

## Delete compute target

Do not delete the compute target if you intend to keep using it for the data drift monitor scheduled runs or otherwise. If the minimum nodes are set to 0, it will automatically scale down.

In [ ]:
# optionally delete the compute target
#compute_target.delete()

## Next steps

  * See [our documentation](https://aka.ms/datadrift) or [Python SDK reference](https://docs.microsoft.com/python/api/overview/azure/ml/intro)
  * [Send requests or feedback](mailto:driftfeedback@microsoft.com) on data drift directly to the team
  * Please open issues with data drift here on GitHub or on StackOverflow if others are likely to run into the same issue